# P-018 SHAP Feature Importance Stability
**Agent OS v3.0 | Materia Arche**

**Decision question:** Are SHAP feature rankings stable, or do they change with different data splits?

NB17 found Jsc, bandgap, Voc as the top-3 drivers of perovskite stability (T80). This packet tests whether those rankings hold across 10 different train/test splits using permutation importance (a model-agnostic proxy for SHAP-style feature importance).

- **Method:** Permutation importance via `sklearn.inspection.permutation_importance`
- **Model:** Locked ExtraTreesRegressor config
- **Splits:** 10 random seeds (0-9), 80/20 train/test
- **Metric:** Rank stability via Kendall tau correlation

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from scipy.stats import kendalltau

# Load data
df = pd.read_csv("perovskite_stability_clean.csv").fillna(0)

FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature", "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF"
]

TARGET = "Stability_PCE_T80"

X = df[FEATURES].copy()
y = np.log1p(df[TARGET].values)

print(f"Dataset: {X.shape[0]} devices, {X.shape[1]} features")
print(f"Target: log1p({TARGET})")

Dataset: 1543 devices, 16 features
Target: log1p(Stability_PCE_T80)


In [2]:
# Cell 3: Permutation importance across 10 train/test splits
ET_PARAMS = dict(
    n_estimators=700,
    max_features="sqrt",
    min_samples_split=3,
    min_samples_leaf=1,
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

n_splits = 10
importance_values = {}   # seed -> array of mean importances
importance_ranks = {}    # seed -> array of ranks (1=most important)

for seed in range(n_splits):
    print(f"Split {seed}/9 (seed={seed})...", end=" ")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=seed
    )
    et = ExtraTreesRegressor(**ET_PARAMS)
    et.fit(X_train, y_train)
    r2 = et.score(X_test, y_test)
    
    perm = permutation_importance(
        et, X_test, y_test,
        n_repeats=10, random_state=seed, n_jobs=-1
    )
    imp = perm.importances_mean
    importance_values[seed] = imp
    # Rank: 1 = most important (highest importance value)
    ranks = len(imp) - np.argsort(np.argsort(imp))
    importance_ranks[seed] = ranks
    
    top3 = [FEATURES[i] for i in np.argsort(imp)[::-1][:3]]
    print(f"R2={r2:.3f}  Top-3: {top3}")

print("\nAll 10 splits complete.")

Split 0/9 (seed=0)... 

R2=0.187  Top-3: ['Perovskite_band_gap', 'Perovskite_thickness', 'Cell_area_measured']
Split 1/9 (seed=1)... 

R2=0.261  Top-3: ['Perovskite_band_gap', 'Cell_area_measured', 'Perovskite_thickness']
Split 2/9 (seed=2)... 

R2=0.141  Top-3: ['Cell_area_measured', 'Perovskite_thickness', 'Perovskite_band_gap']
Split 3/9 (seed=3)... 

R2=0.204  Top-3: ['Perovskite_band_gap', 'Cell_area_measured', 'Perovskite_thickness']
Split 4/9 (seed=4)... 

...

R2=0.186  Top-3: ['Cell_area_measured', 'Perovskite_thickness', 'first_Prvskt_thermal_annealing_time']
Split 6/9 (seed=6)... 

R2=0.254  Top-3: ['Perovskite_band_gap', 'Cell_area_measured', 'first_Prvskt_thermal_annealing_time']
Split 7/9 (seed=7)... 

R2=0.240  Top-3: ['Perovskite_band_gap', 'Perovskite_thickness', 'Cell_area_measured']
Split 8/9 (seed=8)... 

R2=0.065  Top-3: ['Cell_area_measured', 'Perovskite_band_gap', 'first_Prvskt_thermal_annealing_time']
Split 9/9 (seed=9)... 

R2=0.236  Top-3: ['Perovskite_thickness', 'Perovskite_band_gap', 'Cell_area_measured']

All 10 splits complete.


In [3]:
# Cell 4: Stability table — mean/std/min/max rank per feature
rank_df = pd.DataFrame(importance_ranks, index=FEATURES).T  # rows=seeds, cols=features
stability = pd.DataFrame({
    "feature": FEATURES,
    "mean_rank": rank_df.mean().values,
    "std_rank": rank_df.std().values,
    "min_rank": rank_df.min().values,
    "max_rank": rank_df.max().values,
})
stability = stability.sort_values("mean_rank").reset_index(drop=True)

# Flag features that are always top-5
stability["always_top5"] = stability["max_rank"] <= 5

print("="*70)
print("FEATURE IMPORTANCE STABILITY TABLE (sorted by mean rank)")
print("="*70)
print(stability.to_string(index=False))
print()
always_top5 = stability[stability["always_top5"]]["feature"].tolist()
print(f"Features always in top 5 across all 10 splits: {always_top5}")

FEATURE IMPORTANCE STABILITY TABLE (sorted by mean rank)


                            feature  mean_rank  std_rank  min_rank  max_rank  always_top5
                Perovskite_band_gap        1.8  1.032796         1         4         True
                 Cell_area_measured        1.9  0.875595         1         3         True
               Perovskite_thickness        2.7  1.059350         1         4         True
first_Prvskt_thermal_annealing_time        4.1  1.100505         3         6        False
                     JV_default_Jsc        5.2  0.918937         4         7        False
 first_Prvskt_annealing_temperature        6.3  1.946507         4        10        False
                     JV_default_Voc        8.6  3.272783         6        15        False
                                 FA        8.9  2.024846         7        13        False
                                 Sn       10.0  2.357023         7        13        False
                                 Cs       10.5  1.957890         8        14        False
          

In [4]:
# Cell 5: Pairwise Kendall tau rank correlation
from itertools import combinations

taus = []
for s1, s2 in combinations(range(n_splits), 2):
    tau, pval = kendalltau(importance_ranks[s1], importance_ranks[s2])
    taus.append(tau)

mean_tau = np.mean(taus)
std_tau = np.std(taus)
min_tau = np.min(taus)
max_tau = np.max(taus)

print(f"Pairwise Kendall tau across {len(taus)} split pairs:")
print(f"  Mean tau:  {mean_tau:.4f}")
print(f"  Std tau:   {std_tau:.4f}")
print(f"  Min tau:   {min_tau:.4f}")
print(f"  Max tau:   {max_tau:.4f}")
print()
if mean_tau >= 0.80:
    print("=> HIGH stability: rankings are very consistent across splits.")
elif mean_tau >= 0.60:
    print("=> MODERATE stability: rankings are fairly consistent.")
else:
    print("=> LOW stability: rankings shift substantially between splits.")

Pairwise Kendall tau across 45 split pairs:
  Mean tau:  0.6215
  Std tau:   0.1328
  Min tau:   0.3500
  Max tau:   0.8167

=> MODERATE stability: rankings are fairly consistent.


In [5]:
# Cell 6: Top-3 consistency check for Jsc, bandgap, Voc
key_features = ["JV_default_Jsc", "Perovskite_band_gap", "JV_default_Voc"]

print("Top-3 / Top-5 consistency for key features across 10 splits:")
print("-" * 60)

for feat in key_features:
    feat_idx = FEATURES.index(feat)
    ranks_across_splits = [importance_ranks[s][feat_idx] for s in range(n_splits)]
    in_top3 = sum(1 for r in ranks_across_splits if r <= 3)
    in_top5 = sum(1 for r in ranks_across_splits if r <= 5)
    print(f"  {feat:30s}  top-3: {in_top3}/10  top-5: {in_top5}/10  "
          f"ranks: {ranks_across_splits}")

print()

# Overall: are all three in top-3 together?
all_three_top3 = 0
for s in range(n_splits):
    ranks_s = {feat: importance_ranks[s][FEATURES.index(feat)] for feat in key_features}
    if all(r <= 3 for r in ranks_s.values()):
        all_three_top3 += 1

print(f"All three (Jsc, bandgap, Voc) in top 3 together: {all_three_top3}/10 splits")

# Are all three always in top 5?
all_three_top5 = 0
for s in range(n_splits):
    ranks_s = {feat: importance_ranks[s][FEATURES.index(feat)] for feat in key_features}
    if all(r <= 5 for r in ranks_s.values()):
        all_three_top5 += 1

print(f"All three (Jsc, bandgap, Voc) in top 5 together: {all_three_top5}/10 splits")

Top-3 / Top-5 consistency for key features across 10 splits:
------------------------------------------------------------
  JV_default_Jsc                  top-3: 0/10  top-5: 7/10  ranks: [np.int64(4), np.int64(4), np.int64(5), np.int64(6), np.int64(5), np.int64(6), np.int64(5), np.int64(5), np.int64(5), np.int64(7)]
  Perovskite_band_gap             top-3: 9/10  top-5: 10/10  ranks: [np.int64(1), np.int64(1), np.int64(3), np.int64(1), np.int64(2), np.int64(4), np.int64(1), np.int64(1), np.int64(2), np.int64(2)]
  JV_default_Voc                  top-3: 0/10  top-5: 0/10  ranks: [np.int64(6), np.int64(7), np.int64(12), np.int64(7), np.int64(6), np.int64(12), np.int64(9), np.int64(6), np.int64(15), np.int64(6)]

All three (Jsc, bandgap, Voc) in top 3 together: 0/10 splits
All three (Jsc, bandgap, Voc) in top 5 together: 0/10 splits


In [6]:
# Cell 7: Save per-feature per-split importance values and ranks
rows = []
for seed in range(n_splits):
    for i, feat in enumerate(FEATURES):
        rows.append({
            "seed": seed,
            "feature": feat,
            "importance_mean": importance_values[seed][i],
            "rank": importance_ranks[seed][i]
        })

out_df = pd.DataFrame(rows)
out_path = "Packet_P018_SHAP_Stability.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved {out_path}: {out_df.shape[0]} rows ({n_splits} splits x {len(FEATURES)} features)")
print(out_df.head(20))

Saved Packet_P018_SHAP_Stability.csv: 160 rows (10 splits x 16 features)
    seed                              feature  importance_mean  rank
0      0                  Perovskite_band_gap     9.435800e-02     1
1      0                                   Pb    -3.271419e-03    12
2      0                                   Sn     4.361254e-03     8
3      0                                    I    -1.555463e-02    15
4      0                                   Br    -7.935455e-03    14
5      0                                   Cl     1.110223e-16     9
6      0                                   MA    -6.551412e-03    13
7      0                                   FA     5.262117e-03     7
8      0                                   Cs    -1.544572e-03    11
9      0   first_Prvskt_annealing_temperature    -6.598115e-04    10
10     0  first_Prvskt_thermal_annealing_time     2.718325e-02     5
11     0                 Perovskite_thickness     5.091148e-02     2
12     0                   Cel

In [7]:
# Cell 8: Honest status footer
print("=" * 70)
print("P-018 SHAP FEATURE IMPORTANCE STABILITY — VERDICT")
print("=" * 70)
print()

# Criteria
top3_in_top3 = all_three_top3  # from cell 6
top3_in_top5 = all_three_top5  # from cell 6

print(f"Mean pairwise Kendall tau:         {mean_tau:.4f}")
print(f"All 3 key features in top 3:       {top3_in_top3}/10 splits")
print(f"All 3 key features in top 5:       {top3_in_top5}/10 splits")
print()

if top3_in_top3 >= 8 and mean_tau >= 0.80:
    status = "CONFIRMED"
    explanation = ("Top-3 features (Jsc, bandgap, Voc) are the same in >=8/10 splits "
                   "AND mean pairwise Kendall tau >= 0.80. "
                   "SHAP feature rankings are stable across different data splits.")
elif top3_in_top5 >= 7 and mean_tau >= 0.60:
    status = "PROMISING"
    explanation = ("Top-5 features are mostly stable AND mean Kendall tau >= 0.60. "
                   "Rankings are reasonably consistent but not perfectly locked.")
else:
    status = "NEGATIVE"
    explanation = ("Rankings shuffle substantially between splits (tau < 0.60) "
                   "or key features are not consistently top-ranked.")

print(f"Status: {status}")
print(f"Reason: {explanation}")
print()
print("Packet P-018 complete.")

P-018 SHAP FEATURE IMPORTANCE STABILITY — VERDICT

Mean pairwise Kendall tau:         0.6215
All 3 key features in top 3:       0/10 splits
All 3 key features in top 5:       0/10 splits

Status: NEGATIVE
Reason: Rankings shuffle substantially between splits (tau < 0.60) or key features are not consistently top-ranked.

Packet P-018 complete.
